# 10th week's homework - by [Aleksei Novikov](https://www.linkedin.com/in/devnovikov/)

Kubernetes - Deploying lead scoring model to k8s

## Preparation

In this homework, we deploy the lead scoring model from homework 5 to Kubernetes.

Prerequisites:
- Docker installed
- kubectl installed
- kind installed

## Building the Docker Image

First, build the Docker image from the homework 5 directory:

```bash
cd /path/to/machine-learning-zoomcamp/cohorts/2025/05-deployment/homework
docker build -f Dockerfile_full -t zoomcamp-model:3.13.10-hw10 .
```

In [8]:
!docker images | grep zoomcamp-model

zoomcamp-model                     3.13.10-hw10   b5c5447c11af   18 minutes ago   408MB
agrigorev/zoomcamp-model           2025           4a9ecc576ae9   7 weeks ago      121MB


## Question 1: Testing the Model Locally

```bash
docker run -it --rm -p 9696:9696 zoomcamp-model:3.13.10-hw10
```

In another terminal, run the test script (from q6_test).

In [11]:
import requests

url = "http://localhost:9696/predict"
client = {"job": "management", "duration": 400, "poutcome": "success"}

response = requests.post(url, json=client).json()
print(response)

{'conversion_probability': 0.49999999999842815, 'conversion': False}


Output:
```python
{'conversion_probability': 0.49999999999842815, 'conversion': False}
```

**Answer: 0.49**

## Question 2: Kind Version

Check the version of kind installed.

In [12]:
!kind --version

kind version 0.30.0


Output:
```
kind version 0.30.0
```

**Answer: 0.30.0**

## Creating a Kind Cluster

Create a Kubernetes cluster with kind:

```bash
kind create cluster
```

Verify the cluster is running:

In [13]:
!kubectl cluster-info

Kubernetes control plane is running at https://127.0.0.1:49924
CoreDNS is running at https://127.0.0.1:49924/api/v1/namespaces/kube-system/services/kube-dns:dns/proxy

To further debug and diagnose cluster problems, use 'kubectl cluster-info dump'.


Output:
```
Kubernetes control plane is running at https://127.0.0.1:49924
CoreDNS is running at https://127.0.0.1:49924/api/v1/namespaces/kube-system/services/kube-dns:dns/proxy
```

## Question 3: Smallest Deployable Unit

What's the smallest deployable computing unit that we can create and manage in Kubernetes?

Options:
- Node
- Pod
- Deployment
- Service

**Answer: Pod**

## Question 4: Default Service Type

Get the list of running services to see what's the Type of the service that is already running.

In [17]:
!kubectl get services | grep kubernetes

kubernetes            ClusterIP      10.96.0.1      <none>        443/TCP          22h


Output:
```
NAME         TYPE        CLUSTER-IP   EXTERNAL-IP   PORT(S)   AGE
kubernetes   ClusterIP   10.96.0.1    <none>        443/TCP   22h
```

**Answer: ClusterIP**

The default `kubernetes` service that exists in every cluster has the type `ClusterIP`.

## Question 5: Loading Docker Image to Kind

What's the command we need to run to register the docker image with kind?

Options:
- `kind create cluster`
- `kind build node-image`
- `kind load docker-image`
- `kubectl apply`

**Answer: kind load docker-image**

In [ ]:
!kind load docker-image zoomcamp-model:3.13.10-hw10

## Question 6: Creating a Deployment

Create a deployment configuration file `deployment.yaml`:

In [19]:
deployment_yaml = """
apiVersion: apps/v1
kind: Deployment
metadata:
  name: subscription
spec:
  selector:
    matchLabels:
      app: subscription
  replicas: 1
  template:
    metadata:
      labels:
        app: subscription
    spec:
      containers:
      - name: subscription
        image: zoomcamp-model:3.13.10-hw10
        resources:
          requests:
            memory: "64Mi"
            cpu: "100m"
          limits:
            memory: "128Mi"
            cpu: "500m"
        ports:
        - containerPort: 9696
"""

with open('deployment.yaml', 'w') as f:
    f.write(deployment_yaml.strip())
    
print(deployment_yaml)


apiVersion: apps/v1
kind: Deployment
metadata:
  name: subscription
spec:
  selector:
    matchLabels:
      app: subscription
  replicas: 1
  template:
    metadata:
      labels:
        app: subscription
    spec:
      containers:
      - name: subscription
        image: zoomcamp-model:3.13.10-hw10
        resources:
          requests:
            memory: "64Mi"
            cpu: "100m"
          limits:
            memory: "128Mi"
            cpu: "500m"
        ports:
        - containerPort: 9696



The `containerPort` is **9696** because the Dockerfile exposes port 9696 and the FastAPI app runs on that port.

**Answer: 9696**

In [33]:
!kubectl apply -f deployment.yaml

deployment.apps/subscription created


In [34]:
!kubectl get pods

NAME                            READY   STATUS    RESTARTS   AGE
subscription-69c87b4597-hndnv   1/1     Running   0          11s


Output:
```
NAME                            READY   STATUS    RESTARTS   AGE
subscription-69c87b4597-hndnv   1/1     Running   0          11s
```

## Question 7: Creating a Service

Create a service configuration file `service.yaml`:

In [35]:
service_yaml = """
apiVersion: v1
kind: Service
metadata:
  name: subscription
spec:
  type: LoadBalancer
  selector:
    app: subscription
  ports:
  - port: 80
    targetPort: 9696
"""

with open('service.yaml', 'w') as f:
    f.write(service_yaml.strip())
    
print("service.yaml created!")
print(service_yaml)

service.yaml created!

apiVersion: v1
kind: Service
metadata:
  name: subscription
spec:
  type: LoadBalancer
  selector:
    app: subscription
  ports:
  - port: 80
    targetPort: 9696



The selector `app: subscription` must match the label defined in the deployment's pod template.

**Answer: subscription**

In [36]:
!kubectl apply -f service.yaml

service/subscription created


## Testing the Service

Test the service by port-forwarding:

```bash
kubectl port-forward service/subscription 9696:80
```

Then run the test script again to verify it works:

In [39]:
import requests

url = "http://localhost:9696/predict"
client = {"job": "management", "duration": 400, "poutcome": "success"}

response = requests.post(url, json=client).json()
print(response)

{'conversion_probability': 0.49999999999842815, 'conversion': False}


Output:
```python
{'conversion_probability': 0.49999999999842815, 'conversion': False}
```


Config file

```
apiVersion: v1
kind: Service
metadata:
  name: subscription
spec:
  type: LoadBalancer
  selector:
    app: subscription
  ports:
  - port: 80
    targetPort: 9696
```

## Question 8: Autoscaling with HPA

Create a HorizontalPodAutoscaler (HPA) to automatically scale the deployment based on CPU usage.

In [40]:
!kubectl autoscale deployment subscription --name subscription-hpa --cpu-percent=20 --min=1 --max=3

horizontalpodautoscaler.autoscaling/subscription-hpa autoscaled


In [41]:
!kubectl get hpa

NAME               REFERENCE                 TARGETS              MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   cpu: <unknown>/20%   1         3         0          3s


### Load Test Script

Run load test script to increase CPU usage:

In [ ]:
!kubectl get hpa

Output after ~2 minutes of load:
```
NAME               REFERENCE                 TARGETS        MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   cpu: 12%/20%   1         3         3          2m
```

In [ ]:
!kubectl get pods

Output:
```
NAME                            READY   STATUS    RESTARTS   AGE
subscription-69c87b4597-hndnv   1/1     Running   0          3m34s
subscription-69c87b4597-lp5wn   1/1     Running   0          52s
subscription-69c87b4597-xcws4   1/1     Running   0          67s
```

The HPA scaled the deployment from 1 to 3 replicas (the maximum configured) in response to the increased CPU load.

**Answer: 3**

## Summary of Answers

| Question | Answer |
|----------|--------|
| Q1: Conversion probability | 0.49 |
| Q2: Kind version | 0.30.0 |
| Q3: Smallest deployable unit | Pod |
| Q4: Default service type | ClusterIP |
| Q5: Command to load image | kind load docker-image |
| Q6: Container port | 9696 |
| Q7: Selector value | subscription |
| Q8: Max replicas with HPA | 3 |